In [3]:
import requests
import pandas as pd
from functools import reduce

API_KEY = "kMWwLaP6bX0Qwf2zdKR4cE4hVt4r2cDQjpRZJeCg"

##### 1. Reusable fetching function

In [4]:
def fetch_eia_series(series_id, column_name):
    
    url = f"https://api.eia.gov/v2/seriesid/{series_id}"
    
    params = {
        "api_key": API_KEY,
        "length": 5000,
        "offset": 0
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    rows = response.json()["response"]["data"]
    
    df = pd.DataFrame(rows)
    
    df = df[["period", "value"]]
    
    df.columns = ["month", column_name]
    
    df["month"] = pd.to_datetime(df["month"]).dt.to_period("M")
    
    df[column_name] = pd.to_numeric(
        df[column_name],
        errors="coerce"
    )
    
    return df

##### 2. Defining the EIA datasets

In [5]:
series = {

    # US inventories
    "us_crude_inventory":
        "PET.MCRSFUS1.M",

    # Cushing inventories
    "cushing_inventory":
        "PET.MCRST_YCUOK_1.M",

    # Refinery utilization %
    "refinery_utilization":
        "PET.MOPUEUS2.M",

    # US crude imports
    "us_crude_imports":
        "PET.MCRIMUS2.M",

    # US crude exports
    "us_crude_exports":
        "PET.MCREXUS2.M",

    # Global petroleum / liquid fuels consumption
    "global_oil_consumption":
        "STEO.PATC_WORLD.M",

    # OPEC crude oil production
    "opec_crude_production":
        "STEO.COPR_OPEC.M"
}

##### Fetching the Data

In [6]:
datasets = []

for column_name, series_id in series.items():
    
    print("Fetching:", column_name)
    
    df = fetch_eia_series(
        series_id,
        column_name
    )
    
    datasets.append(df)

Fetching: us_crude_inventory
Fetching: cushing_inventory
Fetching: refinery_utilization
Fetching: us_crude_imports
Fetching: us_crude_exports
Fetching: global_oil_consumption
Fetching: opec_crude_production


In [7]:
market_data = reduce(
    lambda left, right:
    pd.merge(left, right, on="month", how="outer"),
    datasets
)

In [8]:
market_data = (
    market_data
    .sort_values("month")
    .reset_index(drop=True)
)

In [9]:
market_data = market_data[
    market_data["month"] >= "2006-09"
]

In [10]:
market_data.head()

,month,us_crude_inventory,cushing_inventory,refinery_utilization,us_crude_imports,us_crude_exports,global_oil_consumption,opec_crude_production
1040,2006-09,214604.0,23016.0,93.0,10710.0,21.0,86.020261,26.4569
1041,2006-10,217164.0,23724.0,87.9,10106.0,37.0,86.376517,26.2080
1042,2006-11,214516.0,26677.0,88.0,9888.0,24.0,87.072642,26.0502
1043,2006-12,204894.0,25895.0,90.6,9555.0,27.0,86.455922,25.5778
1044,2007-01,206218.0,21416.0,88.2,10211.0,9.0,86.524227,25.5085


In [11]:
market_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 256 entries, 1040 to 1295
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype    
---  ------                  --------------  -----    
 0   month                   256 non-null    period[M]
 1   us_crude_inventory      238 non-null    float64  
 2   cushing_inventory       238 non-null    float64  
 3   refinery_utilization    238 non-null    float64  
 4   us_crude_imports        238 non-null    float64  
 5   us_crude_exports        238 non-null    float64  
 6   global_oil_consumption  256 non-null    float64  
 7   opec_crude_production   256 non-null    float64  
dtypes: float64(7), period[M](1)
memory usage: 16.1 KB


In [12]:
market_data.to_csv("market_data.csv", index=False)

In [13]:
market_data.shape

(256, 8)

# Oil Market Intelligence Dataset Overview

This project combines multiple energy-market datasets to study the relationship between crude-oil prices, production, inventories, demand, and macroeconomic conditions.

The final dataset is organized at a **monthly frequency** so that all variables can be compared consistently over time.

## Data Sources

Most variables are collected from the **U.S. Energy Information Administration (EIA)** using its API.

Additional macroeconomic variables such as the U.S. Dollar Index can be collected from **FRED / Federal Reserve data**.

---

## Dataset Features

| Feature | Unit | Meaning |
|---|---|---|
| `month` | YYYY-MM | Month of observation |
| `state` | Categorical | U.S. oil-producing state |
| `state_production_thousand_bpd` | Thousand barrels/day | Average crude-oil production rate for the state during the month |
| `wti_price` | USD/barrel | Monthly average West Texas Intermediate crude-oil price |
| `brent_price` | USD/barrel | Monthly average Brent crude-oil price |
| `brent_wti_spread` | USD/barrel | Difference between Brent and WTI prices |
| `us_crude_inventory` | Thousand barrels | Amount of crude oil stored in the U.S. |
| `cushing_inventory` | Thousand barrels | Crude-oil inventory stored at Cushing, Oklahoma |
| `refinery_utilization` | % | Percentage of U.S. refinery capacity being used |
| `us_crude_imports` | Thousand barrels/day | Average crude-oil imports into the U.S. |
| `us_crude_exports` | Thousand barrels/day | Average crude-oil exports from the U.S. |
| `global_oil_consumption` | Million barrels/day | Global consumption of petroleum and liquid fuels |
| `opec_crude_production` | Million barrels/day | Crude-oil production by OPEC countries |
| `usd_index` | Index value | Measures the strength of the U.S. dollar relative to major currencies |

---

## Meaning of Important Units

### Thousand barrels per day
Used for production, imports, and exports.

Example:

`5800 thousand barrels/day`

means:

`5.8 million barrels per day`

---

### Thousand barrels
Used for inventory variables.

Example:

`35000 thousand barrels`

means:

`35 million barrels of crude oil stored`

Inventory is a **stock variable**, while production, imports, and exports are **flow variables**.

---

### USD per barrel
Used for Brent and WTI crude-oil prices.

Example:

`WTI = 75 USD/barrel`

means one barrel of WTI crude oil is priced at approximately $75.

---

### Percentage
Used for refinery utilization.

Example:

`refinery_utilization = 92%`

means U.S. refineries are operating at approximately 92% of their available capacity.

---

## Data Frequency

The final dataset uses a **monthly frequency**.

Daily Brent and WTI prices are converted into monthly average prices.

For example:

Daily WTI prices during January 2020  
→ average of all available January prices  
→ monthly WTI price for January 2020

State production and most other EIA variables are already available as monthly series.

---

## Dataset Structure

Each row represents one state during one month.

Example:

| month | state | production | WTI | Brent |
|---|---|---:|---:|---:|
| 2020-01 | Texas | 5300 | 57.5 | 63.7 |
| 2020-01 | New Mexico | 1100 | 57.5 | 63.7 |
| 2020-01 | North Dakota | 1400 | 57.5 | 63.7 |

WTI and Brent prices are repeated for every state in the same month because they are market-level variables, while crude production differs by state.

---

## Main Variable Categories

### Supply Variables
- State crude production
- OPEC crude production
- U.S. crude imports
- U.S. crude inventories
- Cushing inventories

### Demand Variables
- Global oil consumption
- Refinery utilization
- U.S. crude exports

### Market Price Variables
- WTI price
- Brent price
- Brent-WTI spread

### Macroeconomic Variable
- U.S. Dollar Index

---

## Objective of the Dataset

The dataset is designed to study:

- crude-oil production trends
- Brent and WTI price movements
- state-level production differences
- price-production relationships
- lagged production responses to oil prices
- inventory and refinery effects
- global supply and demand conditions
- OPEC influence on crude markets
- macroeconomic effects through the U.S. dollar
- crude-oil production forecasting